# AI Model 02 최종 격자 학습 데이터셋 생성

01 산출물과 지역별 주유소 가격 원천을 결합해 AI 학습에 사용할 최종 500m 격자 패널 parquet 하나를 만듭니다.

최종 산출물:
`ROOT_PATH/그리드/grid.parquet`

In [ ]:
# ============================================================
# 0. 공통 경로/환경 설정
# ============================================================
from pathlib import Path
from collections import deque
import gc
import json
import math
import os
import re
import shutil
import subprocess
import sys
import unicodedata

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"

ROOT_DIR = Path(ROOT_PATH)
DATA_COLLECTION_DIR = ROOT_DIR / "data-analysis" / "00_data_collection" / "outputs"
DERIVED_DIR = DATA_COLLECTION_DIR / "derived_data"
GRID_OUTPUT_DIR = ROOT_DIR / "그리드"

DATA_COLLECTION_PATH = str(DATA_COLLECTION_DIR) + "/"
DERIVED_DATA_PATH = str(DERIVED_DIR) + "/"
GRID_OUTPUT_PATH = str(GRID_OUTPUT_DIR) + "/"

GRID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL_SIZE_M = 500
SOURCE_CRS = "EPSG:4326"
WORK_CRS = "EPSG:5179"

START_DATE = "2008-01-01"
END_DATE = None  # None이면 가격 원천/01 산출물 기준 최대 날짜까지 사용

STATION_BATCH_SIZE = 700
DUCKDB_THREADS = 2

STATION_INFLUENCE_BAND_KM = 3.0
STATION_INFLUENCE_CUTOFF_KM = 15.0

FACILITY_DECAY_CONFIG = {
    "storage": {"band_km": 20.0, "cutoff_km": 60.0},
    "agency": {"band_km": 10.0, "cutoff_km": 30.0},
    "factory": {"band_km": 35.0, "cutoff_km": 105.0},
}
FACILITY_CHUNK_SIZE = 5000

FILL_BEFORE_FIRST_OFFICIAL_PRICE = False
CLEAN_TEMP_AFTER_SUCCESS = True

TEMP_ROOT = Path("/content/kff_spatial_grid_build_tmp")
BASE_PARTS_DIR = TEMP_ROOT / "station_additive_parts"
BASE_PANEL_TMP_PATH = TEMP_ROOT / f"grid_station_daily_panel_{CELL_SIZE_M}m_base.parquet"
STATION_KEYS_BY_MONTH_DIR = TEMP_ROOT / "station_keys_by_month"
STATION_INFLUENCE_DIR = TEMP_ROOT / "station_influence_by_month"
FACILITY_STATIC_TMP_PATH = TEMP_ROOT / f"facility_static_{CELL_SIZE_M}m.parquet"
GEO_FLAG_TMP_PATH = TEMP_ROOT / f"geo_flag_land_grid_{CELL_SIZE_M}m.parquet"
OFFICIAL_PRICE_TMP_PATH = TEMP_ROOT / "official_land_price_grid.parquet"

FINAL_PANEL_PATH = GRID_OUTPUT_DIR / f"grid.parquet"

BRAND_ORDER = [
    "SK에너지",
    "GS칼텍스",
    "HD현대오일뱅크",
    "S-OIL",
    "알뜰",
    "NH-OIL",
    "자가상표",
    "기타",
]
SELF_ORDER = ["셀프", "일반", "미상"]

REGION_NAMES = [
    "서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종",
    "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주",
]

REQUIRED_DERIVED_FILES = {
    "land_grid": DERIVED_DIR / f"korea_land_grid_{CELL_SIZE_M}m.parquet",
    "station_location_history": DERIVED_DIR / "station_location_history.csv",
    "station_attribute_history": DERIVED_DIR / "station_attribute_history.csv",
    "facility_points": DERIVED_DIR / "facility_points.csv",
    "official_land_price_grid": DERIVED_DIR / "official_land_price_grid.csv",
}

print(f"ROOT_PATH             = {ROOT_PATH}")
print(f"DATA_COLLECTION_PATH  = {DATA_COLLECTION_PATH}")
print(f"DERIVED_DATA_PATH     = {DERIVED_DATA_PATH}")
print(f"GRID_OUTPUT_PATH      = {GRID_OUTPUT_PATH}")
print(f"FINAL_PANEL_PATH      = {FINAL_PANEL_PATH}")


In [ ]:
# ============================================================
# 1. 패키지/공통 함수
# ============================================================
def ensure_packages():
    packages = [
        ("duckdb", "duckdb"),
        ("pyarrow", "pyarrow"),
        ("scipy", "scipy"),
        ("pyproj", "pyproj"),
    ]
    missing = []
    for import_name, pip_name in packages:
        try:
            __import__(import_name)
        except Exception:
            missing.append(pip_name)
    if missing:
        print(f"[설치] packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


ensure_packages()

import duckdb
from pyproj import Transformer
from scipy.spatial import cKDTree

TO_WORK = Transformer.from_crs(SOURCE_CRS, WORK_CRS, always_xy=True)
TO_WGS = Transformer.from_crs(WORK_CRS, SOURCE_CRS, always_xy=True)


def nfc(value):
    if value is None:
        return value
    return unicodedata.normalize("NFC", str(value))


def normalize_space(value):
    value = "" if pd.isna(value) else str(value)
    value = unicodedata.normalize("NFKC", value)
    return re.sub(r"\s+", " ", value).strip()


def ql(value):
    return "'" + str(value).replace("'", "''") + "'"


def qi(value):
    return '"' + str(value).replace('"', '""') + '"'


def display_df(df, title=None, n=20):
    if title:
        print("\n" + "=" * 100)
        print(title)
    display(df.head(n))


def read_csv_flexible(path, **kwargs):
    path = Path(path)
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as exc:
            last_error = exc
    raise last_error


def clean_number_series(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan, "-": np.nan}),
        errors="coerce",
    )


def parse_date_series(series):
    return pd.to_datetime(series, errors="coerce")


def valid_lonlat(lon, lat):
    lon = pd.to_numeric(lon, errors="coerce")
    lat = pd.to_numeric(lat, errors="coerce")
    return lon.between(120, 135) & lat.between(30, 45)


def ensure_required_inputs():
    missing = []
    for name, path in REQUIRED_DERIVED_FILES.items():
        if not path.exists():
            missing.append({"input": name, "path": str(path)})
    station_root = resolve_station_region_root()
    if not station_root.exists():
        missing.append({"input": "gas_station_prices_by_region", "path": str(station_root)})
    if missing:
        display_df(pd.DataFrame(missing), "[누락 입력]")
        raise FileNotFoundError("02 입력 파일이 부족합니다. 01 산출물과 주유소 가격 원천을 확인하세요.")


def has_station_region_dirs(path):
    path = Path(path)
    if not path.exists():
        return False
    return any((path / region).exists() for region in REGION_NAMES)


def resolve_station_region_root():
    candidates = [
        DATA_COLLECTION_DIR / "gas_station_prices_by_region",
    ]
    for candidate in candidates:
        if has_station_region_dirs(candidate):
            return candidate
    return candidates[0]


def find_region_dir(station_root, region):
    station_root = Path(station_root)
    candidates = [
        station_root / region,
        station_root / nfc(region),
        station_root / unicodedata.normalize("NFD", region),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return station_root / region


def find_station_metadata_path(region_dir):
    region_dir = Path(region_dir)
    candidates = [
        region_dir / "metadata__latlon.json",
        region_dir / "metadata_latlon.json",
        region_dir / "metadata.json",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def reset_temp_dirs():
    if TEMP_ROOT.exists():
        shutil.rmtree(TEMP_ROOT)
    BASE_PARTS_DIR.mkdir(parents=True, exist_ok=True)
    STATION_INFLUENCE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 2. 01 산출물 로드/정리
# ============================================================
ensure_required_inputs()
reset_temp_dirs()

land_grid = pd.read_parquet(REQUIRED_DERIVED_FILES["land_grid"])
land_grid["cell_x"] = land_grid["cell_x"].astype("int32")
land_grid["cell_y"] = land_grid["cell_y"].astype("int32")

station_location_history = read_csv_flexible(REQUIRED_DERIVED_FILES["station_location_history"])
station_attribute_history = read_csv_flexible(REQUIRED_DERIVED_FILES["station_attribute_history"])
facility_points = read_csv_flexible(REQUIRED_DERIVED_FILES["facility_points"])
official_raw = read_csv_flexible(REQUIRED_DERIVED_FILES["official_land_price_grid"])

print(f"[입력] land_grid rows={len(land_grid):,}")
print(f"[입력] station_location_history rows={len(station_location_history):,}")
print(f"[입력] station_attribute_history rows={len(station_attribute_history):,}")
print(f"[입력] facility_points rows={len(facility_points):,}")
print(f"[입력] official_land_price_grid rows={len(official_raw):,}")


def prepare_location_events(df):
    if len(df) == 0:
        return pd.DataFrame(columns=["station_id", "effective_date", "lat", "lon"])
    out = df.copy()
    if "field" in out.columns:
        out = out[out["field"].astype(str).eq("location")].copy()
    out["station_id"] = out["station_id"].astype(str)
    out["effective_date"] = parse_date_series(out["effective_date"])
    out["lat"] = clean_number_series(out["lat"])
    out["lon"] = clean_number_series(out["lon"])
    out = out[valid_lonlat(out["lon"], out["lat"])].copy()
    out = out.dropna(subset=["station_id", "effective_date", "lat", "lon"])
    out = out.sort_values(["station_id", "effective_date"])
    out = out.drop_duplicates(["station_id", "effective_date"], keep="last")
    return out[["station_id", "effective_date", "lat", "lon"]]


def prepare_attribute_events(df, field, out_col):
    if len(df) == 0:
        return pd.DataFrame(columns=["station_id", "effective_date", out_col])
    out = df[df["field"].astype(str).eq(field)].copy()
    out["station_id"] = out["station_id"].astype(str)
    out["effective_date"] = parse_date_series(out["effective_date"])
    out[out_col] = out["value"].map(normalize_space)
    out = out.dropna(subset=["station_id", "effective_date"])
    out = out[out[out_col].astype(str).str.strip().ne("")]
    out = out.sort_values(["station_id", "effective_date"])
    out = out.drop_duplicates(["station_id", "effective_date"], keep="last")
    return out[["station_id", "effective_date", out_col]]


location_events = prepare_location_events(station_location_history)
brand_events = prepare_attribute_events(station_attribute_history, "brand", "brand")
self_events = prepare_attribute_events(station_attribute_history, "is_self", "is_self")

print(f"[정리] location_events rows={len(location_events):,}")
print(f"[정리] brand_events rows={len(brand_events):,}")
print(f"[정리] self_events rows={len(self_events):,}")


In [ ]:

# ============================================================
# 3. 주유소 일별 격자 기본 feature 생성
# ============================================================
def get_station_ids_from_csv(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return []
    cols = read_csv_flexible(csv_path, nrows=0).columns.tolist()
    return [str(c) for c in cols if str(c).strip().lower() != "date"]


def price_part_files(parts_dir):
    parts_dir = Path(parts_dir)
    if not parts_dir.exists() or not parts_dir.is_dir():
        return []
    return sorted(parts_dir.glob("part-*.csv"))


def resolve_price_source(region_dir, fuel):
    region_dir = Path(region_dir)
    candidates = [region_dir / f"{fuel}.csv", region_dir / f"{fuel}.parts"]
    if fuel == "diesel":
        candidates.insert(1, region_dir / "deisel.csv")
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def get_station_ids_from_price_source(source):
    if source is None:
        return []
    source = Path(source)
    if source.is_dir():
        ids = []
        for part_file in price_part_files(source):
            ids.extend(get_station_ids_from_csv(part_file))
        return sorted(set(ids))
    return get_station_ids_from_csv(source)


def wide_subset_to_long(csv_path, value_name, station_ids):
    csv_path = Path(csv_path)
    if not csv_path.exists() or not station_ids:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    header_cols = read_csv_flexible(csv_path, nrows=0).columns.tolist()
    available_ids = [sid for sid in station_ids if sid in header_cols]
    if not available_ids:
        return pd.DataFrame(columns=["date", "station_id", value_name])

    usecols = ["date"] + available_ids
    wide = read_csv_flexible(csv_path, usecols=usecols)
    wide["date"] = parse_date_series(wide["date"])
    wide = wide.dropna(subset=["date"])

    long = (
        wide.melt(id_vars="date", var_name="station_id", value_name=value_name)
        .dropna(subset=[value_name])
        .copy()
    )
    long["station_id"] = long["station_id"].astype(str)
    long[value_name] = clean_number_series(long[value_name])
    long = long.dropna(subset=[value_name])
    return long


def wide_subset_to_long_price_source(source, value_name, station_ids):
    if source is None:
        return pd.DataFrame(columns=["date", "station_id", value_name])
    source = Path(source)
    if source.is_dir():
        frames = [wide_subset_to_long(part_file, value_name, station_ids) for part_file in price_part_files(source)]
        frames = [frame for frame in frames if len(frame) > 0]
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["date", "station_id", value_name])
    return wide_subset_to_long(source, value_name, station_ids)


def add_effective_metadata(base_df, event_df, value_cols):
    if len(base_df) == 0 or len(event_df) == 0:
        for col in value_cols:
            if col not in base_df.columns:
                base_df[col] = np.nan
        return base_df

    event_map = {
        sid: g.sort_values("effective_date").copy()
        for sid, g in event_df.groupby("station_id", sort=False)
    }

    parts = []
    for sid, left_part in base_df.groupby("station_id", sort=False):
        left_part = left_part.sort_values("date").copy()
        right_part = event_map.get(sid)
        if right_part is None or len(right_part) == 0:
            for col in value_cols:
                left_part[col] = np.nan
            parts.append(left_part)
            continue

        merged = pd.merge_asof(
            left_part,
            right_part,
            left_on="date",
            right_on="effective_date",
            direction="backward",
        )
        merged = merged.drop(columns=["effective_date"], errors="ignore")
        if "station_id_x" in merged.columns:
            merged = merged.rename(columns={"station_id_x": "station_id"}).drop(columns=["station_id_y"], errors="ignore")
        parts.append(merged)

    return pd.concat(parts, ignore_index=True) if parts else base_df


def normalize_brand(value):
    s = normalize_space(value)
    upper = s.upper()
    if not s:
        return "기타"
    if "SK" in upper or "에스케이" in s:
        return "SK에너지"
    if "GS" in upper or "지에스" in s:
        return "GS칼텍스"
    if "현대" in s or "HD" in upper or "오일뱅크" in s:
        return "HD현대오일뱅크"
    if "S-OIL" in upper or "SOIL" in upper or "S-OIL" in s or "에쓰" in s:
        return "S-OIL"
    if "알뜰" in s:
        return "알뜰"
    if "NH" in upper or "농협" in s:
        return "NH-OIL"
    if "자가" in s:
        return "자가상표"
    return "기타"


def normalize_self(value):
    s = normalize_space(value)
    if "셀프" in s:
        return "셀프"
    if s:
        return "일반"
    return "미상"


def assign_grid_cell_xy_from_lonlat(df, lon_col="lon", lat_col="lat"):
    out = df.copy()
    x, y = TO_WORK.transform(out[lon_col].to_numpy(), out[lat_col].to_numpy())
    out["cell_x"] = (np.floor(np.asarray(x) / CELL_SIZE_M).astype(np.int64) * CELL_SIZE_M).astype("int32")
    out["cell_y"] = (np.floor(np.asarray(y) / CELL_SIZE_M).astype(np.int64) * CELL_SIZE_M).astype("int32")
    return out


def station_additive_columns():
    cols = [
        "station_count_total",
        "gasoline_station_count",
        "diesel_station_count",
        "gasoline_price_sum",
        "diesel_price_sum",
    ]
    cols += [f"brand_count__{brand}" for brand in BRAND_ORDER]
    cols += [f"self_count__{label}" for label in SELF_ORDER]
    for label in SELF_ORDER:
        cols += [
            f"gasoline_price_sum__{label}",
            f"gasoline_price_count__{label}",
            f"diesel_price_sum__{label}",
            f"diesel_price_count__{label}",
        ]
    return cols


def aggregate_station_day_additive(station_day):
    if len(station_day) == 0:
        return pd.DataFrame(columns=["date", "cell_x", "cell_y"] + station_additive_columns())

    work = station_day.copy()
    work = work[valid_lonlat(work["lon"], work["lat"])].copy()
    if len(work) == 0:
        return pd.DataFrame(columns=["date", "cell_x", "cell_y"] + station_additive_columns())

    work = assign_grid_cell_xy_from_lonlat(work)
    work["brand_cat"] = work["brand"].map(normalize_brand)
    work["self_cat"] = work["is_self"].map(normalize_self)

    work["station_count_total"] = 1
    work["gasoline_station_count"] = work["gasoline_price"].notna().astype("int16")
    work["diesel_station_count"] = work["diesel_price"].notna().astype("int16")
    work["gasoline_price_sum"] = work["gasoline_price"].fillna(0.0).astype("float64")
    work["diesel_price_sum"] = work["diesel_price"].fillna(0.0).astype("float64")

    for brand in BRAND_ORDER:
        work[f"brand_count__{brand}"] = (work["brand_cat"] == brand).astype("int16")
    for label in SELF_ORDER:
        self_mask = work["self_cat"] == label
        gas_present = self_mask & work["gasoline_price"].notna()
        diesel_present = self_mask & work["diesel_price"].notna()
        work[f"self_count__{label}"] = self_mask.astype("int16")
        work[f"gasoline_price_sum__{label}"] = np.where(gas_present, work["gasoline_price"], 0.0)
        work[f"gasoline_price_count__{label}"] = gas_present.astype("int16")
        work[f"diesel_price_sum__{label}"] = np.where(diesel_present, work["diesel_price"], 0.0)
        work[f"diesel_price_count__{label}"] = diesel_present.astype("int16")

    agg = (
        work.groupby(["date", "cell_x", "cell_y"], observed=True)[station_additive_columns()]
        .sum()
        .reset_index()
    )

    int_cols = [
        c for c in agg.columns
        if c.startswith("station_count")
        or c.endswith("_station_count")
        or c.startswith("brand_count__")
        or c.startswith("self_count__")
        or c.endswith("_count__셀프")
        or c.endswith("_count__일반")
        or c.endswith("_count__미상")
    ]
    for col in int_cols:
        agg[col] = agg[col].astype("int32")
    return agg


def process_station_region_to_temp(region_dir, seen_station_ids, part_no_start):
    region_dir = Path(region_dir)
    gasoline_source = resolve_price_source(region_dir, "gasoline")
    diesel_source = resolve_price_source(region_dir, "diesel")

    gas_ids = set(get_station_ids_from_price_source(gasoline_source))
    diesel_ids = set(get_station_ids_from_price_source(diesel_source))
    all_station_ids = sorted((gas_ids | diesel_ids) - seen_station_ids)
    if not all_station_ids:
        print(f"[region] {region_dir.name}: 처리할 신규 station_id 없음")
        return part_no_start

    seen_station_ids.update(all_station_ids)
    total_batches = math.ceil(len(all_station_ids) / STATION_BATCH_SIZE)
    print(f"[region] {region_dir.name}: station_id={len(all_station_ids):,}, batches={total_batches:,}")

    part_no = part_no_start
    for batch_idx, start in enumerate(range(0, len(all_station_ids), STATION_BATCH_SIZE), start=1):
        batch_ids = all_station_ids[start:start + STATION_BATCH_SIZE]
        print(f"  - batch {batch_idx:,}/{total_batches:,} ids={len(batch_ids):,}")

        gas_long = wide_subset_to_long_price_source(gasoline_source, "gasoline_price", batch_ids)
        diesel_long = wide_subset_to_long_price_source(diesel_source, "diesel_price", batch_ids)
        if len(gas_long) == 0 and len(diesel_long) == 0:
            continue

        station_day = pd.merge(gas_long, diesel_long, on=["date", "station_id"], how="outer")
        if START_DATE:
            station_day = station_day[station_day["date"] >= pd.Timestamp(START_DATE)]
        if END_DATE:
            station_day = station_day[station_day["date"] <= pd.Timestamp(END_DATE)]

        station_day = add_effective_metadata(station_day, location_events, ["lat", "lon"])
        station_day = add_effective_metadata(station_day, brand_events, ["brand"])
        station_day = add_effective_metadata(station_day, self_events, ["is_self"])

        for col in ["lat", "lon", "brand", "is_self"]:
            if col not in station_day.columns:
                station_day[col] = np.nan

        additive = aggregate_station_day_additive(station_day)
        if len(additive) > 0:
            part_no += 1
            out_path = BASE_PARTS_DIR / f"station_additive_part_{part_no:05d}.parquet"
            additive.to_parquet(out_path, index=False, compression="zstd")

        del gas_long, diesel_long, station_day, additive
        gc.collect()

    return part_no


station_root = resolve_station_region_root()
seen_station_ids = set()
part_no = 0

print(f"[주유소 가격 입력 root] {station_root}")
for region in REGION_NAMES:
    region_dir = find_region_dir(station_root, region)
    if not region_dir.exists():
        print(f"[주의] 지역 폴더 없음: {region_dir}")
        continue
    part_no = process_station_region_to_temp(region_dir, seen_station_ids, part_no)

station_part_files = sorted(BASE_PARTS_DIR.glob("*.parquet"))
print(f"[station additive temp parts] {len(station_part_files):,}")
if not station_part_files:
    raise RuntimeError("주유소 격자 additive temp parquet가 생성되지 않았습니다.")


In [ ]:
# ============================================================
# 4. 주유소 base panel 생성
# ============================================================
def build_base_panel_from_parts(part_glob, land_grid_df, output_path):
    output_path = Path(output_path)
    if output_path.exists():
        output_path.unlink()

    land_grid_duck = land_grid_df[[
        "grid_id", "cell_x", "cell_y", "center_lon", "center_lat"
    ]].copy()

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")
    con.register("land_grid", land_grid_duck)

    additive_cols = station_additive_columns()
    sum_exprs = ",\n                ".join([f"SUM({qi(c)}) AS {qi(c)}" for c in additive_cols])

    brand_exprs = ",\n        ".join([
        f"CAST(COALESCE(a.{qi(f'brand_count__{brand}')}, 0) AS INTEGER) AS {qi(f'brand_count__{brand}')}"
        for brand in BRAND_ORDER
    ])
    self_count_exprs = ",\n        ".join([
        f"CAST(COALESCE(a.{qi(f'self_count__{label}')}, 0) AS INTEGER) AS {qi(f'self_count__{label}')}"
        for label in SELF_ORDER
    ])
    self_mean_exprs = []
    for label in SELF_ORDER:
        self_mean_exprs.append(
            f"CAST(a.{qi(f'gasoline_price_sum__{label}')} / NULLIF(a.{qi(f'gasoline_price_count__{label}')}, 0) AS FLOAT) "
            f"AS {qi(f'gasoline_price_mean__{label}')}"
        )
        self_mean_exprs.append(
            f"CAST(a.{qi(f'diesel_price_sum__{label}')} / NULLIF(a.{qi(f'diesel_price_count__{label}')}, 0) AS FLOAT) "
            f"AS {qi(f'diesel_price_mean__{label}')}"
        )
    self_mean_exprs = ",\n        ".join(self_mean_exprs)

    sql = f"""
    COPY (
        WITH additive AS (
            SELECT
                date,
                cell_x,
                cell_y,
                {sum_exprs}
            FROM read_parquet({ql(str(part_glob))})
            GROUP BY date, cell_x, cell_y
        )
        SELECT
            a.date,
            lg.grid_id,
            a.cell_x,
            a.cell_y,
            lg.center_lon,
            lg.center_lat,
            CAST(COALESCE(a.station_count_total, 0) AS INTEGER) AS station_count_total,
            CAST(COALESCE(a.gasoline_station_count, 0) AS INTEGER) AS gasoline_station_count,
            CAST(COALESCE(a.diesel_station_count, 0) AS INTEGER) AS diesel_station_count,
            CAST(a.gasoline_price_sum / NULLIF(a.gasoline_station_count, 0) AS FLOAT) AS gasoline_price_mean,
            CAST(a.diesel_price_sum / NULLIF(a.diesel_station_count, 0) AS FLOAT) AS diesel_price_mean,
            {brand_exprs},
            {self_count_exprs},
            {self_mean_exprs}
        FROM additive a
        LEFT JOIN land_grid lg
        USING (cell_x, cell_y)
        WHERE lg.grid_id IS NOT NULL
    ) TO {ql(str(output_path))}
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()


build_base_panel_from_parts(BASE_PARTS_DIR / "*.parquet", land_grid, BASE_PANEL_TMP_PATH)
print(f"[temp 저장] base panel: {BASE_PANEL_TMP_PATH}")


In [ ]:
# ============================================================
# 5. 주유소 주변 영향력 계산
# ============================================================
def compute_station_neighbor_influence(cell_x, cell_y, counts, band_km=3.0, cutoff_km=15.0):
    n = len(counts)
    if n <= 1:
        return np.zeros(n, dtype="float32")

    coords = np.column_stack([
        cell_x.astype("float64") + CELL_SIZE_M / 2.0,
        cell_y.astype("float64") + CELL_SIZE_M / 2.0,
    ])
    band_m = band_km * 1000.0
    cutoff_m = cutoff_km * 1000.0

    tree = cKDTree(coords)
    coo = tree.sparse_distance_matrix(tree, cutoff_m, output_type="coo_matrix")
    if coo.nnz == 0:
        return np.zeros(n, dtype="float32")

    mask = coo.row != coo.col
    if mask.sum() == 0:
        return np.zeros(n, dtype="float32")

    weights = np.exp(-coo.data[mask] / band_m) * counts[coo.col[mask]]
    influence = np.bincount(coo.row[mask], weights=weights, minlength=n)
    return influence.astype("float32")


def build_station_influence_month_files(base_panel_path):
    if STATION_KEYS_BY_MONTH_DIR.exists():
        shutil.rmtree(STATION_KEYS_BY_MONTH_DIR)
    if STATION_INFLUENCE_DIR.exists():
        shutil.rmtree(STATION_INFLUENCE_DIR)
    STATION_KEYS_BY_MONTH_DIR.mkdir(parents=True, exist_ok=True)
    STATION_INFLUENCE_DIR.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")
    con.execute(f"""
        COPY (
            SELECT
                date,
                cell_x,
                cell_y,
                station_count_total,
                EXTRACT(year FROM date)::INTEGER AS year,
                EXTRACT(month FROM date)::INTEGER AS month
            FROM read_parquet({ql(str(base_panel_path))})
        ) TO {ql(str(STATION_KEYS_BY_MONTH_DIR))}
        (FORMAT PARQUET, PARTITION_BY (year, month), COMPRESSION ZSTD)
    """)
    con.close()

    month_dirs = sorted(STATION_KEYS_BY_MONTH_DIR.glob("year=*/month=*"))
    if not month_dirs:
        raise RuntimeError("월별 station influence 입력 파일이 생성되지 않았습니다.")

    for month_dir in month_dirs:
        year_value = int(month_dir.parent.name.split("=")[-1])
        month_value = int(month_dir.name.split("=")[-1])
        print(f"[station influence] {year_value:04d}-{month_value:02d}")

        df_month = pd.read_parquet(month_dir)
        if len(df_month) == 0:
            continue

        parts = []
        total_dates = df_month["date"].nunique()
        for idx, (dt, g) in enumerate(df_month.groupby("date", sort=True), start=1):
            infl = compute_station_neighbor_influence(
                cell_x=g["cell_x"].to_numpy(),
                cell_y=g["cell_y"].to_numpy(),
                counts=g["station_count_total"].to_numpy(dtype="float64"),
                band_km=STATION_INFLUENCE_BAND_KM,
                cutoff_km=STATION_INFLUENCE_CUTOFF_KM,
            )
            part = g[["date", "cell_x", "cell_y"]].copy()
            part["station_neighbor_influence"] = infl
            parts.append(part)
            if idx % 20 == 0 or idx == total_dates:
                print(f"  - {idx:,}/{total_dates:,} dates")

        out_path = STATION_INFLUENCE_DIR / f"station_influence_{year_value:04d}_{month_value:02d}.parquet"
        pd.concat(parts, ignore_index=True).to_parquet(out_path, index=False, compression="zstd")
        del df_month, parts
        gc.collect()


build_station_influence_month_files(BASE_PANEL_TMP_PATH)
print(f"[temp 저장] station influence dir: {STATION_INFLUENCE_DIR}")


In [ ]:
# ============================================================
# 6. 시설 영향력 static feature 생성
# ============================================================
def prepare_facility_points(df):
    if len(df) == 0:
        return pd.DataFrame(columns=["facility_type", "lon", "lat", "x_m", "y_m", "cell_x", "cell_y"])
    out = df.copy()
    out["lon"] = clean_number_series(out["lon"])
    out["lat"] = clean_number_series(out["lat"])
    out = out[valid_lonlat(out["lon"], out["lat"])].copy()
    out = out[out["facility_type"].isin(["storage", "agency", "factory"])].copy()
    if len(out) == 0:
        return pd.DataFrame(columns=["facility_type", "lon", "lat", "x_m", "y_m", "cell_x", "cell_y"])

    x, y = TO_WORK.transform(out["lon"].to_numpy(), out["lat"].to_numpy())
    out["x_m"] = np.asarray(x, dtype="float64")
    out["y_m"] = np.asarray(y, dtype="float64")
    out["cell_x"] = (np.floor(out["x_m"] / CELL_SIZE_M).astype(np.int64) * CELL_SIZE_M).astype("int32")
    out["cell_y"] = (np.floor(out["y_m"] / CELL_SIZE_M).astype(np.int64) * CELL_SIZE_M).astype("int32")
    return out


def compute_decay_sum(target_xy, source_xy, band_km, cutoff_km, chunk_size=5000):
    n_target = target_xy.shape[0]
    score = np.zeros(n_target, dtype="float64")
    if source_xy.shape[0] == 0:
        return score.astype("float32")

    band_m = float(band_km) * 1000.0
    cutoff_m = float(cutoff_km) * 1000.0
    sx = source_xy[:, 0][None, :]
    sy = source_xy[:, 1][None, :]

    for start in range(0, n_target, chunk_size):
        end = min(start + chunk_size, n_target)
        tx = target_xy[start:end, 0][:, None]
        ty = target_xy[start:end, 1][:, None]
        dist = np.hypot(tx - sx, ty - sy)
        weight = np.exp(-dist / band_m)
        weight[dist > cutoff_m] = 0.0
        score[start:end] = weight.sum(axis=1)

    return score.astype("float32")


def build_facility_static_features(land_grid_df, facility_df, output_path):
    out = land_grid_df[[
        "grid_id", "cell_x", "cell_y", "center_x", "center_y", "center_lon", "center_lat"
    ]].copy()
    fac = prepare_facility_points(facility_df)
    print(f"[시설 feature] valid facility rows={len(fac):,}")

    if len(fac) > 0:
        tmp = fac.copy()
        tmp["facility_count_total"] = 1
        tmp["facility_storage_count"] = (tmp["facility_type"] == "storage").astype("int16")
        tmp["facility_factory_count"] = (tmp["facility_type"] == "factory").astype("int16")
        tmp["facility_agency_count"] = (tmp["facility_type"] == "agency").astype("int16")
        counts = (
            tmp.groupby(["cell_x", "cell_y"], observed=True)
            .agg(
                facility_count_total=("facility_count_total", "sum"),
                facility_storage_count=("facility_storage_count", "sum"),
                facility_factory_count=("facility_factory_count", "sum"),
                facility_agency_count=("facility_agency_count", "sum"),
            )
            .reset_index()
        )
        out = out.merge(counts, on=["cell_x", "cell_y"], how="left")
    else:
        for col in [
            "facility_count_total",
            "facility_storage_count",
            "facility_factory_count",
            "facility_agency_count",
        ]:
            out[col] = 0

    for col in [
        "facility_count_total",
        "facility_storage_count",
        "facility_factory_count",
        "facility_agency_count",
    ]:
        out[col] = out[col].fillna(0).astype("int32")

    target_xy = out[["center_x", "center_y"]].to_numpy(dtype="float64")
    for facility_type, out_prefix in [("storage", "storage"), ("agency", "agency"), ("factory", "factory")]:
        src = fac.loc[fac["facility_type"] == facility_type, ["x_m", "y_m"]].to_numpy(dtype="float64")
        cfg = FACILITY_DECAY_CONFIG[facility_type]
        print(f"[시설 영향력] {facility_type}: source={len(src):,}, band={cfg['band_km']}km, cutoff={cfg['cutoff_km']}km")
        out[f"{out_prefix}_influence"] = compute_decay_sum(
            target_xy=target_xy,
            source_xy=src,
            band_km=cfg["band_km"],
            cutoff_km=cfg["cutoff_km"],
            chunk_size=FACILITY_CHUNK_SIZE,
        )

    out = out[[
        "cell_x", "cell_y",
        "facility_count_total",
        "facility_storage_count",
        "facility_factory_count",
        "facility_agency_count",
        "storage_influence",
        "agency_influence",
        "factory_influence",
    ]]
    out.to_parquet(output_path, index=False, compression="zstd")


build_facility_static_features(land_grid, facility_points, FACILITY_STATIC_TMP_PATH)
print(f"[temp 저장] facility static: {FACILITY_STATIC_TMP_PATH}")


In [ ]:
# ============================================================
# 7. 섬/육지 연결 컴포넌트 flag 생성
# ============================================================
def connected_components_on_grid(cell_df, cell_size_m=500):
    coords = list(zip(cell_df["cell_x"].astype(int), cell_df["cell_y"].astype(int)))
    coord_set = set(coords)
    visited = set()
    components = []
    offsets = [
        (-cell_size_m, -cell_size_m), (-cell_size_m, 0), (-cell_size_m, cell_size_m),
        (0, -cell_size_m), (0, cell_size_m),
        (cell_size_m, -cell_size_m), (cell_size_m, 0), (cell_size_m, cell_size_m),
    ]

    for start in coords:
        if start in visited:
            continue
        queue = deque([start])
        visited.add(start)
        comp = []
        while queue:
            cur = queue.popleft()
            comp.append(cur)
            cx, cy = cur
            for dx, dy in offsets:
                nb = (cx + dx, cy + dy)
                if nb in coord_set and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        components.append(comp)
    return components


def build_geo_flags(land_grid_df, output_path):
    base = land_grid_df[["cell_x", "cell_y"]].drop_duplicates().copy()
    components = connected_components_on_grid(base, cell_size_m=CELL_SIZE_M)
    component_sizes = np.array([len(comp) for comp in components], dtype=int)
    mainland_idx = int(component_sizes.argmax())

    rows = []
    for comp_idx, comp in enumerate(components):
        is_island = 0 if comp_idx == mainland_idx else 1
        for x, y in comp:
            rows.append((x, y, comp_idx, is_island))

    geo = pd.DataFrame(rows, columns=["cell_x", "cell_y", "land_component_id", "is_island"])
    geo["cell_x"] = geo["cell_x"].astype("int32")
    geo["cell_y"] = geo["cell_y"].astype("int32")
    geo["land_component_id"] = geo["land_component_id"].astype("int32")
    geo["is_island"] = geo["is_island"].astype("int8")
    geo["is_sea"] = np.int8(0)
    geo.to_parquet(output_path, index=False, compression="zstd")

    summary = pd.DataFrame({
        "land_component_id": np.arange(len(component_sizes), dtype=int),
        "grid_count": component_sizes,
        "is_mainland_component": (np.arange(len(component_sizes)) == mainland_idx).astype(int),
    }).sort_values("grid_count", ascending=False)
    display_df(summary, "[land component summary]", n=20)
    print(f"[geo flags] mainland_component_id={mainland_idx}, components={len(component_sizes):,}")


build_geo_flags(land_grid, GEO_FLAG_TMP_PATH)
print(f"[temp 저장] geo flags: {GEO_FLAG_TMP_PATH}")


In [ ]:
# ============================================================
# 8. 공시지가 grid 정리
# ============================================================
def parse_official_price_cols(columns):
    out = []
    for col in columns:
        if re.match(r"^p_\d{8}$", str(col)):
            out.append(str(col))
    return sorted(out, key=lambda x: x.replace("p_", ""))


def date_key_to_sql_date(date_key):
    date_key = str(date_key)
    return f"{date_key[:4]}-{date_key[4:6]}-{date_key[6:8]}"


def prepare_official_price_grid(raw, output_path):
    required = {"cell_x", "cell_y"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"공시지가 필수 컬럼 누락: {missing}")

    price_cols = parse_official_price_cols(raw.columns)
    if not price_cols:
        raise ValueError("p_YYYYMMDD 형태의 공시지가 snapshot 컬럼이 없습니다.")

    official = raw[["cell_x", "cell_y"] + price_cols].copy()
    official["cell_x"] = official["cell_x"].astype("int32")
    official["cell_y"] = official["cell_y"].astype("int32")
    dup_count = int(official[["cell_x", "cell_y"]].duplicated().sum())
    if dup_count > 0:
        raise ValueError(f"공시지가 cell_x/cell_y 중복: {dup_count:,}건")

    rename_cols = {col: f"official_land_price__{col.replace('p_', '')}" for col in price_cols}
    official = official.rename(columns=rename_cols)
    official_price_cols = list(rename_cols.values())
    for col in official_price_cols:
        official[col] = clean_number_series(official[col]).astype("float32")

    official.to_parquet(output_path, index=False, compression="zstd")
    return official_price_cols


def build_official_asof_sql(official_price_cols, fill_before_first=False):
    date_keys = [col.replace("official_land_price__", "") for col in official_price_cols]
    price_when_parts = []
    source_date_when_parts = []

    for i in range(len(date_keys) - 1, -1, -1):
        dt_key = date_keys[i]
        sql_dt = date_key_to_sql_date(dt_key)
        candidate_cols = official_price_cols[: i + 1][::-1]
        coalesce_expr = "COALESCE(" + ", ".join([f"o.{qi(col)}" for col in candidate_cols]) + ")"

        source_parts = []
        for col in candidate_cols:
            col_dt = col.replace("official_land_price__", "")
            source_parts.append(f"WHEN o.{qi(col)} IS NOT NULL THEN DATE {ql(date_key_to_sql_date(col_dt))}")
        source_expr = "CASE " + " ".join(source_parts) + " ELSE NULL END"

        price_when_parts.append(f"WHEN CAST(p.date AS DATE) >= DATE {ql(sql_dt)} THEN {coalesce_expr}")
        source_date_when_parts.append(f"WHEN CAST(p.date AS DATE) >= DATE {ql(sql_dt)} THEN {source_expr}")

    if fill_before_first:
        first_col = official_price_cols[0]
        first_dt = first_col.replace("official_land_price__", "")
        price_else = f"o.{qi(first_col)}"
        source_else = (
            f"CASE WHEN o.{qi(first_col)} IS NOT NULL THEN DATE {ql(date_key_to_sql_date(first_dt))} ELSE NULL END"
        )
    else:
        price_else = "NULL"
        source_else = "NULL"

    price_case = "CASE " + " ".join(price_when_parts) + f" ELSE {price_else} END"
    source_case = "CASE " + " ".join(source_date_when_parts) + f" ELSE {source_else} END"
    return price_case, source_case


official_price_cols = prepare_official_price_grid(official_raw, OFFICIAL_PRICE_TMP_PATH)
official_price_case_sql, official_source_date_case_sql = build_official_asof_sql(
    official_price_cols,
    fill_before_first=FILL_BEFORE_FIRST_OFFICIAL_PRICE,
)
print(f"[temp 저장] official price: {OFFICIAL_PRICE_TMP_PATH}")
print(f"[공시지가 snapshot] {official_price_cols}")


In [ ]:
# ============================================================
# 9. 최종 AI 학습용 격자 panel 하나로 결합
# ============================================================
def build_final_panel():
    if FINAL_PANEL_PATH.exists():
        FINAL_PANEL_PATH.unlink()

    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")

    station_infl_glob = STATION_INFLUENCE_DIR / "*.parquet"

    sql = f"""
    COPY (
        WITH panel AS (
            SELECT * FROM read_parquet({ql(str(BASE_PANEL_TMP_PATH))})
        ),
        infl AS (
            SELECT date, cell_x, cell_y, station_neighbor_influence
            FROM read_parquet({ql(str(station_infl_glob))})
        ),
        fac AS (
            SELECT * FROM read_parquet({ql(str(FACILITY_STATIC_TMP_PATH))})
        ),
        geo AS (
            SELECT * FROM read_parquet({ql(str(GEO_FLAG_TMP_PATH))})
        ),
        official AS (
            SELECT * FROM read_parquet({ql(str(OFFICIAL_PRICE_TMP_PATH))})
        )
        SELECT
            p.date,
            p.grid_id,
            p.cell_x,
            p.cell_y,
            p.center_lon,
            p.center_lat,
            p.station_count_total,
            p.gasoline_station_count,
            p.diesel_station_count,
            p.gasoline_price_mean,
            p.diesel_price_mean,
            {", ".join([f"p.{qi(f'brand_count__{brand}')}" for brand in BRAND_ORDER])},
            {", ".join([f"p.{qi(f'self_count__{label}')}" for label in SELF_ORDER])},
            {", ".join([f"p.{qi(f'gasoline_price_mean__{label}')}, p.{qi(f'diesel_price_mean__{label}')}" for label in SELF_ORDER])},
            CAST(COALESCE(i.station_neighbor_influence, 0.0) AS FLOAT) AS station_neighbor_influence,
            CAST(COALESCE(f.facility_count_total, 0) AS INTEGER) AS facility_count_total,
            CAST(COALESCE(f.facility_storage_count, 0) AS INTEGER) AS facility_storage_count,
            CAST(COALESCE(f.facility_factory_count, 0) AS INTEGER) AS facility_factory_count,
            CAST(COALESCE(f.facility_agency_count, 0) AS INTEGER) AS facility_agency_count,
            CAST(COALESCE(f.storage_influence, 0.0) AS FLOAT) AS storage_influence,
            CAST(COALESCE(f.agency_influence, 0.0) AS FLOAT) AS agency_influence,
            CAST(COALESCE(f.factory_influence, 0.0) AS FLOAT) AS factory_influence,
            CAST(COALESCE(g.is_sea, 0) AS INTEGER) AS is_sea,
            CAST(COALESCE(g.is_island, 0) AS INTEGER) AS is_island,
            CAST(g.land_component_id AS INTEGER) AS land_component_id,
            CAST({official_price_case_sql} AS FLOAT) AS official_land_price,
            CAST({official_source_date_case_sql} AS DATE) AS official_price_source_date
        FROM panel p
        LEFT JOIN infl i USING (date, cell_x, cell_y)
        LEFT JOIN fac f USING (cell_x, cell_y)
        LEFT JOIN geo g USING (cell_x, cell_y)
        LEFT JOIN official o USING (cell_x, cell_y)
    ) TO {ql(str(FINAL_PANEL_PATH))}
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
    con.execute(sql)
    con.close()


build_final_panel()
print(f"[SAVE FINAL] {FINAL_PANEL_PATH}")


In [ ]:
# ============================================================
# 10. 최종 산출물 검증
# ============================================================
def final_panel_summary(final_path):
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCKDB_THREADS};")
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet({ql(str(final_path))})").df()
    summary = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT grid_id) AS unique_grid_count,
            MIN(CAST(date AS DATE)) AS date_min,
            MAX(CAST(date AS DATE)) AS date_max,
            SUM(CASE WHEN gasoline_price_mean IS NOT NULL THEN 1 ELSE 0 END) AS gasoline_target_rows,
            SUM(CASE WHEN diesel_price_mean IS NOT NULL THEN 1 ELSE 0 END) AS diesel_target_rows,
            SUM(CASE WHEN official_land_price IS NULL THEN 1 ELSE 0 END) AS official_land_price_null_rows,
            AVG(station_neighbor_influence) AS avg_station_neighbor_influence,
            AVG(storage_influence) AS avg_storage_influence,
            AVG(agency_influence) AS avg_agency_influence,
            AVG(factory_influence) AS avg_factory_influence
        FROM read_parquet({ql(str(final_path))})
    """).df()
    con.close()

    file_size_mb = Path(final_path).stat().st_size / (1024 * 1024)
    summary.insert(0, "file", Path(final_path).name)
    summary.insert(1, "file_size_mb", round(file_size_mb, 2))
    return schema, summary


schema_df, summary_df = final_panel_summary(FINAL_PANEL_PATH)
display_df(summary_df, "[최종 AI 격자 패널 요약]")
display_df(schema_df, "[최종 AI 격자 패널 schema]", n=80)

if CLEAN_TEMP_AFTER_SUCCESS and TEMP_ROOT.exists():
    shutil.rmtree(TEMP_ROOT)
    print(f"[REMOVE TEMP] {TEMP_ROOT}")

print("\n[AI Model 02 완료]")
print(f"- final_panel: {FINAL_PANEL_PATH}")
print("- permanent_output_count: 1")